# FUL-105 — EDA preliminar de GLDAS, enero de 2005

Este cuaderno trabaja únicamente con `data/raw/nasa/gldas/GLDAS_NOAH025_M.A200501.021.nc4` y su manifiesto local. La grilla de origen es global; la caja `138,-38,153,-24` selecciona coordenadas de una región rectangular y no representa la cuenca Murray–Darling. No calcula series históricas, tendencias ni inferencia.

Ejecutar desde la raíz: `uv run jupyter nbconvert --to notebook --execute research/notebooks/ful_105_eda_gldas_200501.ipynb --output ful_105_eda_gldas_200501.executed.ipynb --output-dir research/notebooks`. Los valores se leen con `decode_cf=False` y `mask_and_scale=False`; los sentinelas faltantes se excluyen solo al calcular QA, sin escala ni offset.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))
from research.gldas_qa import inspect_gldas_sample

DATA = ROOT / 'data/raw/nasa/gldas/GLDAS_NOAH025_M.A200501.021.nc4'
MANIFEST = DATA.with_name(DATA.name + '.manifest.json')
BBOX = (138.0, -38.0, 153.0, -24.0)

if DATA.is_file() and MANIFEST.is_file():
    manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
    sha256 = hashlib.sha256(DATA.read_bytes()).hexdigest()
    assert sha256 == manifest['sha256'], 'SHA-256 no coincide con el manifiesto'
    assert manifest['product']['short_name'] == 'GLDAS_NOAH025_M'
    print({'file': DATA.name, 'bytes': DATA.stat().st_size, 'sha256': sha256, 'granule_id': manifest['granule_id'], 'period': manifest['period'], 'downloaded_at_utc': manifest['downloaded_at_utc']})
    ds, qa = inspect_gldas_sample(DATA, bbox=BBOX)
    print(json.dumps(qa, indent=2, ensure_ascii=False))
else:
    ds = None
    print('EDA bloqueado: falta el NetCDF GLDAS auténtico y/o su manifiesto local. No se sustituyen por datos sintéticos.')

EDA bloqueado: falta el NetCDF GLDAS auténtico y/o su manifiesto local. No se sustituyen por datos sintéticos.


## Inspección y controles QA

El resumen de `research.gldas_qa` informa cobertura y estadísticos sobre la representación numérica almacenada. Los valores `_FillValue`/`missing_value` y NaN no cuentan como válidos; el cero sí. Si el archivo contiene `scale_factor` o `add_offset`, los números resumidos siguen siendo crudos y no equivalen automáticamente a unidades físicas.

In [2]:
if ds is not None:
    da = ds['SoilMoi0_10cm_inst']
    print('Dimensiones:', dict(ds.sizes))
    print('Coordenadas:', {name: {'size': ds[name].size, 'min': float(ds[name].min()), 'max': float(ds[name].max()), 'units': ds[name].attrs.get('units')} for name in ('lat', 'lon')})
    print('Atributos de variable:', {k: str(v) for k, v in da.attrs.items()})
    print('Variables candidatas de QA (requieren interpretación documental):', [name for name in ds.data_vars if any(token in name.lower() for token in ('qa', 'quality', 'flag', 'error'))])
    print('Nota: origen global:', list(da.shape), '| selección rectangular:', qa['subset_shape'], '| agregado de cuenca: no calculado')

## Mapas exploratorios y distribución

Los mapas presentan la variable almacenada en su unidad/representación declarada y marcan los sentinelas como faltantes. El panel regional es solo el bbox. La máscara de cuenca requiere un polígono revisado y una decisión de rasterización/ponderación por área.

In [3]:
if ds is not None:
    da = ds['SoilMoi0_10cm_inst'].isel(time=0) if 'time' in ds['SoilMoi0_10cm_inst'].dims else ds['SoilMoi0_10cm_inst']
    raw = np.asarray(da.values, dtype=float)
    missing = ~np.isfinite(raw)
    for key in ('_FillValue', 'missing_value'):
        sentinel = da.attrs.get(key)
        if sentinel is not None:
            for value in np.asarray(sentinel).reshape(-1):
                missing |= np.isnan(raw) if np.isnan(value) else raw == value
    plotted = np.ma.array(raw, mask=missing)
    west, south, east, north = BBOX
    lat = ds.lat.values
    lon = ds.lon.values
    lat_slice = slice(south, north) if lat[0] <= lat[-1] else slice(north, south)
    lon_slice = slice(west, east) if lon[0] <= lon[-1] else slice(east, west)
    regional = da.sel(lat=lat_slice, lon=lon_slice)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    global_plot = axes[0].pcolormesh(ds.lon, ds.lat, plotted, shading='auto', cmap='viridis')
    axes[0].set(title='GLDAS global — SoilMoi0_10cm_inst (crudo)', xlabel='Longitud (°E)', ylabel='Latitud (°N)')
    fig.colorbar(global_plot, ax=axes[0], label=da.attrs.get('units', 'unidad no informada'))
    regional_raw = np.asarray(regional.values, dtype=float).squeeze()
    regional_mask = ~np.isfinite(regional_raw)
    for key in ('_FillValue', 'missing_value'):
        sentinel = da.attrs.get(key)
        if sentinel is not None:
            for value in np.asarray(sentinel).reshape(-1):
                regional_mask |= np.isnan(regional_raw) if np.isnan(value) else regional_raw == value
    local_plot = axes[1].pcolormesh(regional.lon, regional.lat, np.ma.array(regional_raw, mask=regional_mask), shading='auto', cmap='viridis')
    axes[1].set(title='Selección rectangular Murray–Darling (no es máscara de cuenca)', xlabel='Longitud (°E)', ylabel='Latitud (°N)')
    fig.colorbar(local_plot, ax=axes[1], label=da.attrs.get('units', 'unidad no informada'))
    plt.show()
    valid_values = raw[~missing]
    if valid_values.size:
        plt.figure(figsize=(8, 4))
        plt.hist(valid_values, bins=40)
        plt.title('Distribución global de valores válidos almacenados — un mes')
        plt.xlabel(f"Valor crudo ({da.attrs.get('units', 'unidad no informada')})")
        plt.ylabel('Celdas')
        plt.show()

## Interpretación y límites

Una única instantánea mensual permite verificar la estructura del producto, faltantes y distribución espacial básica. No sustenta tendencias, regresiones, significancia, causalidad ni caracterización de anomalías. Próximo paso: validar el polígono Murray–Darling, el tratamiento de celdas parciales y el peso de área con Lucas (FUL-104); coordinar con Alaín (FUL-106) disponibilidad/QA de las otras colecciones y manifiestos. Para comparar variables harán falta unidades, calendarios, resoluciones, máscaras y cobertura temporal armonizados.